# 1

### Nota: 10

En este reto de programación, se te pide aplicar las funciones esenciales de la librería Pandas para manipular, limpiar y fusionar datos. El objetivo principal es que practiques el manejo de valores faltantes (NaN), el tratamiento de columnas categóricas y la fusión de DataFrames.

In [137]:
import pandas as pd

data = [
    {"nombre": "Juan", "edad": 25, "ciudad": "Madrid", "ingresos": 3000},
    {"nombre": "Ana", "edad": None, "ciudad": None, "ingresos": 2500},
    {"nombre": "Pedro", "edad": 30, "ciudad": "Barcelona", "ingresos": None},
    {"nombre": None, "edad": None, "ciudad": "Valencia", "ingresos": None},
    {"nombre": "Luisa", "edad": 35, "ciudad": "Madrid", "ingresos": None}
]

# Creamos el DataFrame
df = pd.DataFrame(data)

# Vemos cuántos valores nulos hay por columna
print(df.isna().sum())

# Rellenamos los valores nulos de las columnas edad e ingresos usando la media
df = df.fillna({"edad": df.edad.mean(), "ingresos": df.ingresos.mean()})

# Ahora las columnas categóricas usando la moda
df = df.fillna({"nombre": df.nombre.mode(), "ciudad": df.ciudad.mode().iloc[0]})

#Convertimos la columna ciudad a una columna de tipo "category"
df["ciudad"] = df.ciudad.astype("category")

#Agregamos la nueva fila mediante concat
new_row = pd.DataFrame([{"nombre": "Nuevo", "edad": 40, "ciudad": "Sevilla", "ingresos": 4000}])
df_ = pd.concat([df, new_row], axis = 0)

# Volvemos a convertir la columna ciudad en "category"
df_["ciudad"] = df_.ciudad.astype("category")

# Verificamos las categorías
print(df_.ciudad.cat.categories)

# Establecemos las categorías de la columna ciudad
df_["ciudad"] = df_.ciudad.cat.set_categories(['Madrid', 'Barcelona', 'Valencia', 'Sevilla'])

# Ahora las reordenamos las categorías
df_["ciudad"] = df_.ciudad.cat.reorder_categories(['Madrid', 'Barcelona', 'Valencia', 'Sevilla'])

# Creamos el segundo DataFrame
data_extra = [
    {"nombre": "Juan", "profesion": "Ingeniero"},
    {"nombre": "Ana", "profesion": "Médico"},
    {"nombre": "Pedro", "profesion": "Abogado"},
    {"nombre": "Luisa", "profesion": "Diseñadora"},
    {"nombre": "Nuevo", "profesion": "Artista"}
]
df2 = pd.DataFrame(data_extra)

#Hacemos el merge entre los dos DataFrames
df_merged = df_.merge(df2, on = "nombre", how = "inner", validate = "many_to_one")

#Lo imprimimos
print(df_merged)

nombre      1
edad        2
ciudad      1
ingresos    3
dtype: int64
Index(['Barcelona', 'Madrid', 'Sevilla', 'Valencia'], dtype='str')
  nombre  edad     ciudad  ingresos   profesion
0   Juan  25.0     Madrid    3000.0   Ingeniero
1    Ana  30.0     Madrid    2500.0      Médico
2  Pedro  30.0  Barcelona    2750.0     Abogado
3  Pedro  30.0   Valencia    2750.0     Abogado
4  Luisa  35.0     Madrid    2750.0  Diseñadora
5  Nuevo  40.0    Sevilla    4000.0     Artista


# 2

### Nota: 8.52

En este reto, métete en la piel de un analista de datos que trabaja para una empresa de eventos deportivos. La empresa ha recopilado datos de sus eventos, los aficionados que asistieron y las promociones publicitarias realizadas para atraer a más asistentes. Tu tarea será procesar y analizar estos datos utilizando Python y la librería Pandas, combinando la información para extraer 'insights' clave que ayuden a mejorar la planificación de futuros eventos y promociones.

In [138]:
import numpy as np
import pandas as pd

work_dir = "../media/0_lab3/"
eventos_dir = "eventos.csv"
aficionados_dir = "aficionados.csv"
promociones_dir = "promociones.csv"

Cargamos los datos, los volcamos en DataFrames, y hacemos un head para 

In [139]:
eventos = pd.read_csv(work_dir + eventos_dir)
aficionados = pd.read_csv(work_dir + aficionados_dir)
promociones = pd.read_csv(work_dir + promociones_dir)

In [140]:
eventos.head()

,id_evento,nombre_evento,fecha_evento,ubicación_evento,asistentes_totales
0,1,Evento 1,2023-01-01,Ciudad C,1292
1,2,Evento 2,2023-01-02,Ciudad A,9629
2,3,Evento 3,2023-01-03,Ciudad A,254
3,4,Evento 4,2023-01-04,Ciudad D,8914
4,5,Evento 5,2023-01-05,Ciudad C,7647


In [141]:
aficionados.head()

,id_aficionado,nombre_aficionado,edad,género,ubicación_residencia,id_evento
0,1,Aficionado 1,45,F,Ciudad B,41
1,2,Aficionado 2,20,F,Ciudad D,3
2,3,Aficionado 3,20,F,Ciudad A,25
3,4,Aficionado 4,36,F,Ciudad B,33
4,5,Aficionado 5,36,M,Ciudad C,14


In [142]:
promociones.head()

,id_promocion,medio_publicitario,presupuesto,fecha_inicio,fecha_fin,id_evento
0,1,Radio,8880,2023-01-01,2023-01-06,23
1,2,Radio,15767,2023-01-06,2023-01-11,23
2,3,Internet,6804,2023-01-11,2023-01-16,3
3,4,Redes Sociales,5520,2023-01-16,2023-01-21,7
4,5,Radio,19598,2023-01-21,2023-01-26,17


Veamos los valores nulos por columna de cada DataFrame.

In [143]:
eventos.isna().sum()

id_evento             0
nombre_evento         0
fecha_evento          0
ubicación_evento      0
asistentes_totales    0
dtype: int64

In [144]:
aficionados.isna().sum()

id_aficionado           0
nombre_aficionado       0
edad                    0
género                  0
ubicación_residencia    0
id_evento               0
dtype: int64

In [145]:
promociones.isna().sum()

id_promocion          0
medio_publicitario    0
presupuesto           0
fecha_inicio          0
fecha_fin             0
id_evento             0
dtype: int64

No hay ningún valor nulo, así que podemos trabajar sin problema. Vamos a combinar los tres DataFrames en 2 distintos, uno que relacione eventos y aficionados, y otro que relacione eventos y promociones. Hacemos un merge mediante inner.

In [146]:
ev_af = eventos.merge(aficionados, on = "id_evento", how = "inner")
ev_prom = eventos.merge(promociones, on = "id_evento", how = "inner")

Con estos dos DataFrame podemos contestar a las siguientes preguntas:

### ¿Cuál es el evento con mayor asistencia total?

Como es una columna del DF de eventos, simplemente es encontrar el valor máximo.

In [147]:
ev_max = eventos.loc[eventos.asistentes_totales == eventos.asistentes_totales.max()].iloc[0]
print(f"El evento con mayor asistencia total es el {ev_max.nombre_evento.lower()} con un total de {ev_max.asistentes_totales} asistentes")

El evento con mayor asistencia total es el evento 29 con un total de 9879 asistentes


### ¿Qué rango de edad asiste más frecuentemente a los eventos?

Usando *pd.cut* definimos el rango, y luego vemos cuantos hay en cada grupo usando el id_aficionado, aunque se podría usar cualquier columna.

In [148]:
ev_af["Edad Discreta"] = pd.cut(ev_af.edad, bins = [0, 18, 35, 65, np.inf], labels = ["Menor (0 - 18)", "Jóven (19 - 35)", "Adulto (36 - 65)", "Mayor (65 - )"])

print("Número de aficionados por rango de edad")
cuenta_rangos = ev_af.groupby("Edad Discreta").id_aficionado.count()
print(cuenta_rangos)

Número de aficionados por rango de edad
Edad Discreta
Menor (0 - 18)       2
Jóven (19 - 35)     22
Adulto (36 - 65)    26
Name: id_aficionado, dtype: int64


### ¿Qué medio publicitario ha generado mayor impacto en términos de asistencia?

Creamos la nueva columna *Impacto Promoción* y agrupamos por el medio publicitario sumando el impacto de la promoción.

In [149]:
ev_prom["Impacto Promoción"] = ev_prom.asistentes_totales / ev_prom.presupuesto
ev_prom.groupby("medio_publicitario")["Impacto Promoción"].sum()

medio_publicitario
Internet          3.300438
Radio             6.363767
Redes Sociales    5.784982
TV                5.573428
Name: Impacto Promoción, dtype: float64

El medio publicitario que ha generado mayor impacto ha sido la radio.

### ¿Cuál es la ubicación de residencia que más aficionados aporta a los eventos?

En el DataFrame con eventos y aficionados, hacemos un group by.

In [150]:
ev_af.groupby("ubicación_residencia").id_evento.count()

ubicación_residencia
Ciudad A    15
Ciudad B    12
Ciudad C    12
Ciudad D    11
Name: id_evento, dtype: int64

La ubicación que más aporta a los eventos es la Ciudad A.

Guardamos el DataFrame resultante con las transformaciones y analisis.

In [152]:
df_res = pd.DataFrame({
    "Pregunta": [
        "¿Cuál es el evento con mayor asistencia total?",
        "¿Cuál es el rango de edad que asiste más frecuentemente a los eventos?",
        "¿Qué medio publicitario ha generado mayor impacto en términos de asistencia?",
        "¿Cuál es la ubicación de residencia que más aficionados aporta a los eventos?"
    ],
    "DataFrame usado" : [
        "eventos",
        "eventos + aficionados",
        "eventos + promociones",
        "eventos + aficionados"
    ],
    "Respuesta": [
        "El evento con mayor asistencia total es el evento 29 con un total de 9879 asistentes.",
        "Número de aficionados por rango de edad: " + str(cuenta_rangos),
        "El medio publicitario que ha generado mayor impacto ha sido la radio.",
        "La ubicación que más aporta a los eventos es la Ciudad A."
    ]
})

df_res.to_csv(work_dir + "reporte_eventos.csv")